# Stochastic Fairness Python script

## Author: Samuel Rodriguez-Gonzalez, PhD
### Last modified: 2025-08-01

This notebook is a thin driver. The SVI data, Gurobi helpers, the EV/EEVS, WS and DE model builders, and the experiment driver logic all live in `../src/`:

- `svi_data.py` - Social Vulnerability Index ranges used to sample fairness weights.
- `gurobi_utils.py` - shared Gurobi parameter configuration and results/summary-statistics helpers.
- `paths.py` - resolves the repo-level `results/` output directory regardless of the notebook's working directory.
- `ev_eevs_model.py` - `EV_and_EEVS` (Expected Value + Expectation of the EV Solution).
- `ws_model.py` - `WS` (wait-and-see).
- `de_model.py` - `DE` (deterministic equivalent).
- `run_experiments.py` - loads network parameters, runs all three models across scarcity levels, and reports EVPI/VSS metrics.

Outputs (CSV/TXT/log files) are written to the repo-level `results/` folder, not the notebook's working directory.

In [1]:
import sys
from pathlib import Path

SRC_DIR = str(Path.cwd().parent / 'src')
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import run_experiments


In [2]:
# Runs EV/EEVS, WS and DE across the default scarcity levels (0.25, 0.5, 0.75, 1)
# and writes the per-run CSV/TXT outputs, mirroring the original inline script.
# results_by_scarcity = run_experiments.run_all()


In [2]:
# import run_formulation_b_pilot as p
# stage1 = p.run_stage1_degenerate_check()      # quick correctness check
# p.run_full_case_study()

## Jensen gap: ex-post Formulation B evaluation from the existing DE results

Formulation B (log-of-expected-utility) was found not to converge in practical time at full 3-network scale
(see docs/dossier.md, Section 11), so instead of solving its nonconvex MIQCP, this reads the DE
(deterministic equivalent) CSV already saved by the cell above and evaluates Formulation B's
log(E[phi_i**alpha_i]) objective directly on that same allocation - no re-solve required. Reports the
"Jensen gap" (log-of-expectation minus expectation-of-log) both in total and per demand node.

In [3]:
# Reads results/DE_scarce_*.csv (produced by the run_all() cell above) and computes the ex-post
# Formulation B Jensen gap - no Gurobi solve involved. See src/jensen_gap_from_de_csv.py.
import jensen_gap_from_de_csv as jg
from svi_data import SVI

csv_path = jg.find_latest_de_csv()
print(f'Reading DE solution from: {csv_path}')
jensen_result = jg.jensen_gap_from_de_csv(csv_path, svi=SVI)

print(f"Scenarios: {jensen_result['num_scenarios']}")
print(f"OF_A (DE, recomputed check):                                  {jensen_result['OF_A_DE_check']:.6f}")
print(f"OF_B (ex-post log-of-expected-utility on DE's own allocation): {jensen_result['OF_B_expost']:.6f}")
print(f"Jensen gap (OF_B_expost - OF_A_DE):                            {jensen_result['jensen_gap']:.6f}")
jensen_result['per_node'].head(10)

Reading DE solution from: C:\Projects\Random personal\Stochastic fairness 2026\stochasticFairness\results\DE_scarce_0.75_utility_type_power_num_scenarios_150_equal_probabilities_True_nat_log_linear_pieces_length_1.csv
Scenarios: 150
OF_A (DE, recomputed check):                                  121.279165
OF_B (ex-post log-of-expected-utility on DE's own allocation): 126.399616
Jensen gap (OF_B_expost - OF_A_DE):                            5.120451


,network,commodity,node,E_log_util_A,log_E_util_B,node_jensen_gap,svi_width
0,Gas,Gas,Gas_node_9,1.985291,2.561874,0.576583,0.9993
1,Water,Water,Water_node_18,1.483306,1.934113,0.450807,0.9993
2,Power,Power,Power_node_19,1.693455,2.142020,0.448564,0.9993
3,Water,Water,Water_node_22,2.134802,2.523597,0.388795,0.7913
4,Power,Power,Power_node_14,1.306555,1.670203,0.363648,0.9993
5,Power,Power,Power_node_36,1.212515,1.502870,0.290355,0.9993
6,Power,Power,Power_node_34,1.167835,1.375896,0.208061,0.9993
7,Power,Power,Power_node_31,1.070089,1.251770,0.181681,0.6909
8,Water,Water,Water_node_20,1.003840,1.166684,0.162843,0.9993
9,Power,Power,Power_node_33,0.915628,1.058622,0.142994,0.6909
